# Gans Scooter – City Data Engineering Pipeline

This notebook implements the **Gans Scooter ETL data pipeline**:

**Wikipedia → Python → MySQL**  
**OpenWeather API → Python → MySQL**  
**RapidAPI / AeroDataBox → Python → MySQL**  
**MySQL → Python validation**

The pipeline collects data for:

- Berlin
- Hamburg
- Munich
- Frankfurt
- Stuttgart

> **Important:** Keep API keys and database passwords in a `.env` file. Do not put credentials directly into the notebook or commit `.env` to GitHub.


## 1. Install dependencies

Run this cell once if the required Python packages are not installed.

```bash
pip install requests beautifulsoup4 mysql-connector-python python-dotenv
```


In [ ]:
# Uncomment and run this cell if the packages are not installed.
# %pip install requests beautifulsoup4 mysql-connector-python python-dotenv


## 2. Imports and configuration

The notebook loads API keys and MySQL credentials from `.env`.

Example `.env`:

```text
OPENWEATHER_API_KEY=your_openweather_api_key
RAPIDAPI_KEY=your_rapidapi_key

MYSQL_HOST=localhost
MYSQL_USER=root
MYSQL_PASSWORD=your_mysql_password
MYSQL_DATABASE=gans_cities
```


In [ ]:
import os
import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import mysql.connector
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD", ""),
    "database": os.getenv("MYSQL_DATABASE", "gans_cities"),
}

CITIES = {
    "Berlin": {
        "wikipedia": "https://en.wikipedia.org/wiki/Berlin",
        "icao": "EDDB",
    },
    "Hamburg": {
        "wikipedia": "https://en.wikipedia.org/wiki/Hamburg",
        "icao": "EDDH",
    },
    "Munich": {
        "wikipedia": "https://en.wikipedia.org/wiki/Munich",
        "icao": "EDDM",
    },
    "Frankfurt": {
        "wikipedia": "https://en.wikipedia.org/wiki/Frankfurt",
        "icao": "EDDF",
    },
    "Stuttgart": {
        "wikipedia": "https://en.wikipedia.org/wiki/Stuttgart",
        "icao": "EDDS",
    },
}

WIKIPEDIA_HEADERS = {
    "User-Agent": "GansScooterDataPipeline/1.0 (educational project)"
}

RAPIDAPI_HOST = "aerodatabox.p.rapidapi.com"

RAPIDAPI_HEADERS = {
    "X-RapidAPI-Key": RAPIDAPI_KEY or "",
    "X-RapidAPI-Host": RAPIDAPI_HOST,
}

print("Configuration loaded.")
print("Cities:", ", ".join(CITIES.keys()))


## 3. Validate configuration

In [ ]:
def validate_configuration():
    missing = []

    if not OPENWEATHER_API_KEY:
        missing.append("OPENWEATHER_API_KEY")

    if not RAPIDAPI_KEY:
        missing.append("RAPIDAPI_KEY")

    if not MYSQL_CONFIG["user"]:
        missing.append("MYSQL_USER")

    if missing:
        raise RuntimeError(
            "Missing configuration values: " + ", ".join(missing)
        )

    print("Configuration validation passed.")


def get_mysql_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)


validate_configuration()


## 4. Wikipedia – Extract city and population data

Wikipedia is scraped with `Requests` and `BeautifulSoup`.

The pipeline extracts:

- city
- country
- latitude
- longitude
- population
- date gathered


In [ ]:
def parse_coordinate(value):
    value = value.strip()

    match = re.search(r"-?\d+(?:\.\d+)?", value)

    if not match:
        return None

    coordinate = float(match.group(0))
    value_upper = value.upper()

    if "S" in value_upper or "W" in value_upper:
        coordinate = -abs(coordinate)

    return coordinate


def extract_population(infobox):
    for row in infobox.find_all("tr"):
        header = row.find("th")
        value = row.find("td")

        if not header or not value:
            continue

        key = header.get_text(" ", strip=True).lower()

        if key != "population":
            continue

        text = value.get_text(" ", strip=True)
        matches = re.findall(r"\d[\d,.\s]*", text)

        for raw_number in matches:
            number = (
                raw_number
                .replace(",", "")
                .replace(".", "")
                .replace(" ", "")
            )

            if number.isdigit():
                return int(number)

    return None


def scrape_city(city_name, wikipedia_url):
    print(f"[Wikipedia] Scraping {city_name}...")

    response = requests.get(
        wikipedia_url,
        headers=WIKIPEDIA_HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    infobox = soup.find("table", class_="infobox")

    if not infobox:
        raise ValueError(
            f"Wikipedia infobox not found for {city_name}"
        )

    latitude_element = soup.find(class_="latitude")
    longitude_element = soup.find(class_="longitude")

    latitude = (
        parse_coordinate(latitude_element.get_text(strip=True))
        if latitude_element
        else None
    )

    longitude = (
        parse_coordinate(longitude_element.get_text(strip=True))
        if longitude_element
        else None
    )

    population = extract_population(infobox)

    if population is None:
        raise ValueError(f"Population not found for {city_name}")

    city_data = {
        "city": city_name,
        "country": "Germany",
        "latitude": latitude,
        "longitude": longitude,
        "population": population,
        "date_gathered": datetime.now().date(),
    }

    print(f"  Population: {population:,}")
    print(f"  Coordinates: {latitude}, {longitude}")

    return city_data


In [ ]:
city_data_list = []

for city_name, config in CITIES.items():
    try:
        city_data = scrape_city(
            city_name,
            config["wikipedia"],
        )
        city_data_list.append(city_data)

    except Exception as error:
        print(f"ERROR scraping {city_name}: {error}")

print(f"\nCities successfully collected: {len(city_data_list)}")


## 5. Load cities and populations into MySQL

In [ ]:
def load_city_data(city_data_list):
    print("[MySQL] Loading cities and populations...")

    connection = get_mysql_connection()
    cursor = connection.cursor()

    try:
        for city_data in city_data_list:
            city_name = city_data["city"]

            cursor.execute(
                '''
                SELECT city_id
                FROM cities
                WHERE city = %s
                ''',
                (city_name,),
            )

            result = cursor.fetchone()

            if result:
                city_id = result[0]

                cursor.execute(
                    '''
                    UPDATE cities
                    SET country = %s,
                        latitude = %s,
                        longitude = %s
                    WHERE city_id = %s
                    ''',
                    (
                        city_data["country"],
                        city_data["latitude"],
                        city_data["longitude"],
                        city_id,
                    ),
                )

                print(f"  Updated city: {city_name}")

            else:
                cursor.execute(
                    '''
                    INSERT INTO cities (
                        city,
                        country,
                        latitude,
                        longitude
                    )
                    VALUES (%s, %s, %s, %s)
                    ''',
                    (
                        city_data["city"],
                        city_data["country"],
                        city_data["latitude"],
                        city_data["longitude"],
                    ),
                )

                city_id = cursor.lastrowid
                print(f"  Inserted city: {city_name}")

            cursor.execute(
                '''
                INSERT INTO populations (
                    city_id,
                    population,
                    date_gathered
                )
                VALUES (%s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    population = VALUES(population)
                ''',
                (
                    city_id,
                    city_data["population"],
                    city_data["date_gathered"],
                ),
            )

        connection.commit()
        print("Cities and populations loaded successfully.")

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


load_city_data(city_data_list)


## 6. OpenWeather API – Extract weather data

Current weather is collected using each city's latitude and longitude.

The pipeline stores:

- temperature
- feels-like temperature
- rain
- wind
- snow
- sunrise
- sunset
- weather description
- observation datetime


In [ ]:
def get_weather_data(city_name, latitude, longitude):
    print(f"[OpenWeather] Collecting {city_name}...")

    url = "https://api.openweathermap.org/data/3.0/onecall"

    params = {
        "lat": latitude,
        "lon": longitude,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric",
        "exclude": "minutely,hourly,daily,alerts",
    }

    response = requests.get(
        url,
        params=params,
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    current = data["current"]

    rain_data = current.get("rain", {})
    snow_data = current.get("snow", {})
    weather = current.get("weather", [])

    weather_description = (
        weather[0].get("description")
        if weather
        else None
    )

    utc = ZoneInfo("UTC")

    observation_datetime = datetime.fromtimestamp(
        current["dt"],
        tz=utc
    ).replace(tzinfo=None)

    sunrise = datetime.fromtimestamp(
        current["sunrise"],
        tz=utc
    ).replace(tzinfo=None)

    sunset = datetime.fromtimestamp(
        current["sunset"],
        tz=utc
    ).replace(tzinfo=None)

    result = {
        "city": city_name,
        "observation_datetime": observation_datetime,
        "temp": current.get("temp"),
        "feels_like": current.get("feels_like"),
        "rain": rain_data.get("1h", 0),
        "wind": current.get("wind_speed"),
        "snow": snow_data.get("1h", 0),
        "sunrise": sunrise,
        "sunset": sunset,
        "weather_description": weather_description,
    }

    print(
        f"  {result['temp']} °C | "
        f"{result['weather_description']}"
    )

    return result


In [ ]:
def load_weather_data():
    print("[MySQL] Loading weather data...")

    connection = get_mysql_connection()
    cursor = connection.cursor(dictionary=True)

    try:
        cursor.execute(
            '''
            SELECT
                city_id,
                city,
                latitude,
                longitude
            FROM cities
            WHERE city IN (
                'Berlin',
                'Hamburg',
                'Munich',
                'Frankfurt',
                'Stuttgart'
            )
            '''
        )

        cities = cursor.fetchall()

        for city in cities:
            if (
                city["latitude"] is None
                or city["longitude"] is None
            ):
                print(
                    f"Skipping {city['city']}: "
                    "missing coordinates"
                )
                continue

            weather = get_weather_data(
                city["city"],
                city["latitude"],
                city["longitude"],
            )

            cursor.execute(
                '''
                INSERT INTO weather (
                    observation_datetime,
                    temp,
                    feels_like,
                    rain,
                    wind,
                    snow,
                    city_id,
                    sunrise,
                    sunset,
                    weather_description
                )
                VALUES (
                    %s, %s, %s, %s, %s,
                    %s, %s, %s, %s, %s
                )
                ''',
                (
                    weather["observation_datetime"],
                    weather["temp"],
                    weather["feels_like"],
                    weather["rain"],
                    weather["wind"],
                    weather["snow"],
                    city["city_id"],
                    weather["sunrise"],
                    weather["sunset"],
                    weather["weather_description"],
                ),
            )

        connection.commit()
        print("Weather data loaded successfully.")

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


load_weather_data()


## 7. Airports

The pipeline maps each city to its main airport using ICAO codes.

| City | ICAO |
|---|---|
| Berlin | EDDB |
| Hamburg | EDDH |
| Munich | EDDM |
| Frankfurt | EDDF |
| Stuttgart | EDDS |


In [ ]:
def load_airports():
    print("[MySQL] Loading airports...")

    connection = get_mysql_connection()
    cursor = connection.cursor(dictionary=True)

    try:
        for city_name, config in CITIES.items():
            cursor.execute(
                '''
                SELECT city_id
                FROM cities
                WHERE city = %s
                ''',
                (city_name,),
            )

            city = cursor.fetchone()

            if not city:
                print(f"City not found: {city_name}")
                continue

            cursor.execute(
                '''
                INSERT INTO airports (
                    city_id,
                    icao
                )
                VALUES (%s, %s)
                ON DUPLICATE KEY UPDATE
                    city_id = VALUES(city_id)
                ''',
                (
                    city["city_id"],
                    config["icao"],
                ),
            )

            print(f"  {city_name}: {config['icao']}")

        connection.commit()
        print("Airports loaded successfully.")

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


load_airports()


## 8. RapidAPI / AeroDataBox – Extract flight data

Flight data is collected through **AeroDataBox via RapidAPI**.

The notebook collects arriving flights for the five airports.

The target date is **tomorrow**, based on Europe/Berlin time.


In [ ]:
def get_flight_data(airport_icao, target_date):
    print(
        f"[RapidAPI/AeroDataBox] "
        f"Collecting arrivals for {airport_icao}..."
    )

    all_flights = []

    time_windows = [
        ("00:00", "11:59"),
        ("12:00", "23:59"),
    ]

    for start_time, end_time in time_windows:
        url = (
            "https://aerodatabox.p.rapidapi.com/"
            f"flights/airports/icao/"
            f"{airport_icao}/"
            f"{target_date}T{start_time}/"
            f"{target_date}T{end_time}"
        )

        params = {
            "withLeg": "true",
            "direction": "Arrival",
            "withCancelled": "false",
            "withCodeshared": "true",
            "withCargo": "false",
            "withPrivate": "false",
            "withLocation": "false",
        }

        response = requests.get(
            url,
            headers=RAPIDAPI_HEADERS,
            params=params,
            timeout=30,
        )

        response.raise_for_status()

        data = response.json()

        for flight in data.get("arrivals", []):
            departure = flight.get("departure", {})
            departure_airport = departure.get("airport", {})

            arrival = flight.get("arrival", {})
            scheduled_time = arrival.get("scheduledTime", {})
            scheduled_local = scheduled_time.get("local")

            all_flights.append(
                {
                    "arrival_airport_icao": airport_icao,
                    "departure_airport_icao":
                        departure_airport.get("icao"),
                    "departure_airport_name":
                        departure_airport.get("name"),
                    "flight_number":
                        flight.get("number"),
                    "scheduled_arrival_time":
                        scheduled_local,
                    "data_retrieved_at":
                        datetime.now(
                            ZoneInfo("Europe/Berlin")
                        ).replace(tzinfo=None),
                }
            )

    print(f"  Flights collected: {len(all_flights)}")

    return all_flights


In [ ]:
def load_flight_data():
    print("[MySQL] Loading flight data...")

    berlin_time = ZoneInfo("Europe/Berlin")

    target_date = (
        datetime.now(berlin_time).date()
        + timedelta(days=1)
    )

    print(f"Flight date: {target_date}")

    all_flights = []

    for city_name, config in CITIES.items():
        try:
            flights = get_flight_data(
                config["icao"],
                target_date,
            )
            all_flights.extend(flights)

        except requests.HTTPError as error:
            print(
                f"API error for {city_name}: {error}"
            )

        except Exception as error:
            print(
                f"Error for {city_name}: {error}"
            )

    if not all_flights:
        print("No flights collected.")
        return

    connection = get_mysql_connection()
    cursor = connection.cursor()

    try:
        sql = '''
            INSERT INTO flights (
                arrival_airport_icao,
                departure_airport_icao,
                departure_airport_name,
                flight_number,
                scheduled_arrival_time,
                data_retrieved_at
            )
            VALUES (%s, %s, %s, %s, %s, %s)
        '''

        for flight in all_flights:
            cursor.execute(
                sql,
                (
                    flight["arrival_airport_icao"],
                    flight["departure_airport_icao"],
                    flight["departure_airport_name"],
                    flight["flight_number"],
                    flight["scheduled_arrival_time"],
                    flight["data_retrieved_at"],
                ),
            )

        connection.commit()

        print(
            f"Inserted {len(all_flights)} flights."
        )

    except Exception:
        connection.rollback()
        raise

    finally:
        cursor.close()
        connection.close()


load_flight_data()


## 9. Read data back from MySQL

This section validates the ETL pipeline by querying the database after the load step.

The notebook checks:

1. Cities and populations
2. Weather observations
3. Flight records


In [ ]:
def read_data_back():
    print("=" * 70)
    print("VALIDATION: READING DATA BACK FROM MYSQL")
    print("=" * 70)

    connection = get_mysql_connection()
    cursor = connection.cursor(dictionary=True)

    try:
        print("\n--- CITIES / POPULATIONS ---")

        cursor.execute(
            '''
            SELECT
                c.city_id,
                c.city,
                c.country,
                c.latitude,
                c.longitude,
                p.population,
                p.date_gathered
            FROM cities AS c
            LEFT JOIN populations AS p
                ON c.city_id = p.city_id
            ORDER BY c.city
            '''
        )

        for row in cursor.fetchall():
            print(row)

        print("\n--- WEATHER ---")

        cursor.execute(
            '''
            SELECT
                c.city,
                w.observation_datetime,
                w.temp,
                w.feels_like,
                w.rain,
                w.wind,
                w.snow,
                w.weather_description
            FROM weather AS w
            INNER JOIN cities AS c
                ON w.city_id = c.city_id
            ORDER BY w.observation_datetime DESC
            '''
        )

        for row in cursor.fetchall():
            print(row)

        print("\n--- FLIGHTS ---")

        cursor.execute(
            '''
            SELECT
                flight_number,
                departure_airport_icao,
                departure_airport_name,
                arrival_airport_icao,
                scheduled_arrival_time,
                data_retrieved_at
            FROM flights
            ORDER BY scheduled_arrival_time
            LIMIT 20
            '''
        )

        for row in cursor.fetchall():
            print(row)

    finally:
        cursor.close()
        connection.close()


read_data_back()


## 10. Pipeline summary

The complete ETL workflow is:

```text
Wikipedia
   ↓
Web Scraping
   ↓
City + Population Data
   ↓
MySQL
   ↓
City Coordinates
   ↓
OpenWeather API
   ↓
Weather Data
   ↓
MySQL
   ↓
Airport ICAO Codes
   ↓
RapidAPI / AeroDataBox
   ↓
Flight Data
   ↓
MySQL
   ↓
SQL Queries / Validation
```

### Project skills demonstrated

- Python
- SQL
- MySQL
- ETL
- Web scraping
- REST APIs
- Requests
- BeautifulSoup
- OpenWeather API
- RapidAPI
- AeroDataBox
- Relational database design
- Foreign keys
- Data validation
- Environment-variable management
